# Pulser + Pasqal via qBraid Runtime

This notebook demonstrates how to submit a [Pulser](https://pulser.readthedocs.io/) pulse sequence to a Pasqal device using the qBraid Runtime.

Pulser is an open-source Python library for designing pulse sequences for neutral-atom quantum processors. Here we build a simple sequence using a Rydberg global channel and submit it to the Pasqal EMU-TN tensor-network simulator via the `QbraidProvider`.

In [ ]:
%%capture

%pip install 'qbraid[azure,pulser]'

In [ ]:
import numpy as np
import pulser
from qbraid.runtime import QbraidProvider

provider = QbraidProvider()

input_data = {}

qubits = {
    "q0": (0, 0),
    "q1": (0, 10),
    "q2": (8, 2),
    "q3": (1, 15),
    "q4": (-10, -3),
    "q5": (-8, 5),
}
register = pulser.Register(qubits)

sequence = pulser.Sequence(register, pulser.DigitalAnalogDevice)
sequence.declare_channel("ch0", "rydberg_global")

amp_wf = pulser.BlackmanWaveform(1000, np.pi)
det_wf = pulser.RampWaveform(1000, -5, 5)
pulse = pulser.Pulse(amp_wf, det_wf, 0)
sequence.add(pulse, "ch0")

device = provider.get_device("azure:pasqal:sim:emu-tn")

job = device.run(sequence, shots=1)

In [ ]:
job.status()

In [ ]:
job.wait_for_final_state()

result = job.result()

result.data.measurement_counts